In [3]:
import sys
from pathlib import Path
import torch

print("=== Screening Resume Check ===")
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("Drive accessible:", Path("/content/drive/MyDrive").is_dir())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Screening cannot resume on this CPU runtime.")

print("No files modified. No model inference performed.")

=== Screening Resume Check ===
Python: 3.13.18
PyTorch: 2.14.1+cpu
CUDA build: None
CUDA available: False
Drive accessible: False
Screening cannot resume on this CPU runtime.
No files modified. No model inference performed.


In [4]:
from google.colab import drive
from pathlib import Path
from collections import Counter
import csv
import hashlib
import io
import json

drive.mount("/content/drive")
repo = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")

csv_bytes = (
    repo / "docs/research_reports/notebook05b_checkpoint_01579_v1/"
    "scenario_metrics_v1.csv"
).read_bytes()
if hashlib.sha256(csv_bytes).hexdigest() != (
    "d6412fec7d767d773786560734eccc8484c7d935a9dfe9cc74a49d45a3931b8b"
):
    raise RuntimeError("Checkpoint CSV SHA256 mismatch.")

plan_bytes = (
    repo / "experiments/notebook05b/int8_screening_plan_v1.json"
).read_bytes()
if hashlib.sha256(plan_bytes).hexdigest() != (
    "9981145a30329b326534faaae8fdffc7be911d1169b3869c081de612838caa6b"
):
    raise RuntimeError("Frozen plan SHA256 mismatch.")

rows = list(csv.DictReader(io.StringIO(csv_bytes.decode("utf-8"))))
plan = json.loads(plan_bytes)
layers = [row["name"] for row in plan["inventory"]]

if len(rows) != 1579:
    raise RuntimeError("Unexpected checkpoint row count.")

by_layer = Counter(row["layer"] for row in rows)
by_layer_bit = Counter(
    (row["layer"], int(row["bit"])) for row in rows
)
stratum_counts = [
    by_layer_bit[(layer, bit)]
    for layer in layers for bit in range(8)
]

print("Checkpoint scenarios:", len(rows))
print("Layers sampled:", len(by_layer), "/ 63")
print("Completed samples per layer: min/max =", (
    min(by_layer[layer] for layer in layers),
    max(by_layer[layer] for layer in layers),
))
print("Layer-bit groups sampled:", sum(n > 0 for n in stratum_counts), "/ 504")
print("Layer-bit groups without samples:", sum(n == 0 for n in stratum_counts))
print("Samples per layer-bit group: min/max =", (
    min(stratum_counts), max(stratum_counts)
))
print("Frozen target: 80 scenarios/layer; 10 samples/layer-bit group.")
print("Coverage audit only. No sensitivity ranking inferred.")
print("No files modified. No inference, commit or push performed.")

Mounted at /content/drive
Checkpoint scenarios: 1579
Layers sampled: 63 / 63
Completed samples per layer: min/max = (15, 39)
Layer-bit groups sampled: 494 / 504
Layer-bit groups without samples: 10
Samples per layer-bit group: min/max = (0, 8)
Frozen target: 80 scenarios/layer; 10 samples/layer-bit group.
Coverage audit only. No sensitivity ranking inferred.
No files modified. No inference, commit or push performed.


In [5]:
import math

scenario_lookup = {
    scenario["scenario_id"]: scenario
    for scenario in plan["scenarios"]
}

audited_rows = []
for row in rows:
    scenario = scenario_lookup[row["scenario_id"]]
    delta = float(row["AP50_95_change"])

    if (
        row["layer"] != scenario["layer"]
        or int(row["bit"]) != scenario["bit"]
        or not math.isfinite(delta)
    ):
        raise RuntimeError("Checkpoint scenario identity mismatch.")

    audited_rows.append((delta, row["scenario_id"], scenario))

worst_observed = sorted(
    audited_rows, key=lambda item: (item[0], item[1])
)[:5]

print("Five largest observed AP drops in the incomplete checkpoint")
print("Scenario | Layer | Bit | AP change (percentage points) | INT8 transition")

for delta, scenario_id, scenario in worst_observed:
    print(
        f"{scenario_id} | {scenario['layer']} | {scenario['bit']} | "
        f"{100 * delta:+.6f} | "
        f"{scenario['q_clean']} -> {scenario['q_faulty']}"
    )

print("\nExploratory case selection based on observed AP drops.")
print("These cases do not establish the population worst case or layer ranking.")
print("No files modified. No inference performed.")

Five largest observed AP drops in the incomplete checkpoint
Scenario | Layer | Bit | AP change (percentage points) | INT8 transition
S05B-00731 | model.0.conv | 7 | -8.592369 | -69 -> 59
S05B-00277 | model.0.conv | 7 | -4.993299 | -71 -> 57
S05B-00112 | model.0.conv | 7 | -1.002128 | -28 -> 100
S05B-00529 | model.2.cv2.conv | 7 | -0.715880 | -15 -> 113
S05B-00074 | model.2.cv2.conv | 7 | -0.645306 | 15 -> -113

Exploratory case selection based on observed AP drops.
These cases do not establish the population worst case or layer ranking.
No files modified. No inference performed.


In [6]:
from pathlib import Path
import gzip
import hashlib
import json
from collections import Counter

repo = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
output_dir = repo / "results/notebook05b/int8_screening_v1"
checkpoint_dir = (
    repo / "docs/research_reports/notebook05b_checkpoint_01579_v1"
)
case_id = "S05B-00731"

def sha256(content):
    return hashlib.sha256(content).hexdigest()

def canonical(value):
    return json.dumps(
        value, sort_keys=True, ensure_ascii=False,
        allow_nan=False, separators=(",", ":"),
    ).encode("utf-8")

def verified_record(path, expected_sha):
    content = path.read_bytes()
    if sha256(content) != expected_sha:
        raise RuntimeError(f"Archive SHA256 mismatch: {path.name}")
    envelope = json.loads(gzip.decompress(content))
    payload = envelope["payload"]
    if sha256(canonical(payload)) != envelope["payload_sha256"]:
        raise RuntimeError(f"Payload checksum mismatch: {path.name}")
    return payload

audit_bytes = (checkpoint_dir / "evaluation_audit_v1.json").read_bytes()
if sha256(audit_bytes) != (
    "a7dee0776a260c0d4d0aab0809a1465931bcb92cec41bb8dc172dcdad4d94df6"
):
    raise RuntimeError("Published audit SHA256 mismatch.")
audit = json.loads(audit_bytes)
entry = next(row for row in audit["artifacts"] if row["scenario_id"] == case_id)

baseline_bytes = (
    checkpoint_dir / "screening_clean_evaluation_v1.json"
).read_bytes()
if sha256(baseline_bytes) != (
    "39c18cdf41849dddf97e1455883b835f47ff3a35d1d42b031c44b511c685a6a8"
):
    raise RuntimeError("Published clean evaluation SHA256 mismatch.")
baseline = json.loads(baseline_bytes)["record"]["payload"]

case_evaluation = verified_record(
    output_dir / f"{case_id}_evaluation.json.gz",
    entry["evaluation_sha256_verified_now"],
)
if case_evaluation["prediction_sha256"] != (
    entry["prediction_sha256_recorded_at_execution"]
):
    raise RuntimeError("Fault prediction references disagree.")

case_clean_record = verified_record(
    output_dir / "clean_predictions.json.gz",
    baseline["prediction_sha256"],
)
case_fault_record = verified_record(
    output_dir / f"{case_id}_predictions.json.gz",
    case_evaluation["prediction_sha256"],
)

for label, record in [
    ("clean", case_clean_record),
    ("faulty", case_fault_record),
]:
    if (
        record["kind"] != "predictions"
        or record["session_fingerprint"] != audit["session_fingerprint"]
        or record["plan_sha256"] != audit["plan_sha256"]
    ):
        raise RuntimeError(f"Prediction provenance mismatch: {label}")

    data = record["data"]
    counts = Counter(p["image_id"] for p in data["predictions"])
    if (
        len(data["image_ids"]) != 200
        or len(set(data["image_ids"])) != 200
        or [row["image_id"] for row in data["per_image"]] != data["image_ids"]
        or set(counts) - set(data["image_ids"])
        or any(
            row["count"] != counts[row["image_id"]]
            for row in data["per_image"]
        )
    ):
        raise RuntimeError(f"Prediction execution record mismatch: {label}")

if (
    case_clean_record["scenario"] is not None
    or case_fault_record["scenario"] != case_evaluation["scenario"]
    or case_clean_record["data"]["image_ids"]
    != case_fault_record["data"]["image_ids"]
):
    raise RuntimeError("Matched scenario/image identity mismatch.")

print("Clean and faulty archive/payload checksums: PASS")
print("Matched image IDs and execution counts: PASS")
print("Scenario:", case_evaluation["scenario"])
print("Clean AP50:95:", baseline["metrics"]["AP50_95"])
print("Faulty AP50:95:", case_evaluation["metrics"]["AP50_95"])
print("Clean prediction count:", len(case_clean_record["data"]["predictions"]))
print("Faulty prediction count:", len(case_fault_record["data"]["predictions"]))
print("Prediction counts alone are not accuracy measures.")
print("No inference or file changes. No commit or push.")

Clean and faulty archive/payload checksums: PASS
Matched image IDs and execution counts: PASS
Scenario: {'bit': 7, 'clean_weight': -0.18211929500102997, 'faulty_weight': 0.15572519600391388, 'flat_index': 475, 'layer': 'model.0.conv', 'q_clean': -69, 'q_faulty': 59, 'replicate': 1, 'scale': 0.002639410085976124, 'scenario_id': 'S05B-00731'}
Clean AP50:95: 0.4842769592793853
Faulty AP50:95: 0.3983532699011195
Clean prediction count: 19809
Faulty prediction count: 22303
Prediction counts alone are not accuracy measures.
No inference or file changes. No commit or push.


In [7]:
from pathlib import Path
import contextlib
import copy
import hashlib
import io
import json
import numpy as np
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

root = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance")
repo = root / "code"
destination = (
    repo / "results/notebook05b/case_studies/"
    "S05B-00731_cocoeval_diagnostics_v1.json"
)
if destination.exists():
    raise FileExistsError("Case-study report already exists; inspect it first.")

split_bytes = (
    repo / "experiments/notebook05/coco2017_split_manifest_v1.json"
).read_bytes()
if hashlib.sha256(split_bytes).hexdigest() != (
    "143c3273b2bf0cd6393cfe148a74e48fbafb74e770bd3fbdf3f14222355978ee"
):
    raise RuntimeError("Split manifest identity mismatch.")

split = json.loads(split_bytes)
annotation_path = (
    root / "datasets/coco2017/annotations/instances_val2017.json"
)
if hashlib.sha256(annotation_path.read_bytes()).hexdigest() != (
    split["annotation_sha256"]
):
    raise RuntimeError("Ground-truth annotation identity mismatch.")

image_ids = case_clean_record["data"]["image_ids"]
metric_names = [
    "AP50_95", "AP50", "AP75", "AP_small", "AP_medium", "AP_large",
    "AR1", "AR10", "AR100", "AR_small", "AR_medium", "AR_large",
]

with contextlib.redirect_stdout(io.StringIO()):
    ground_truth = COCO(str(annotation_path))

def evaluate_saved(record, expected_metrics):
    predictions = copy.deepcopy(record["data"]["predictions"])

    with contextlib.redirect_stdout(io.StringIO()):
        detections = ground_truth.loadRes(predictions)
        evaluator = COCOeval(ground_truth, detections, "bbox")
        evaluator.params.imgIds = image_ids
        evaluator.params.maxDets = [1, 10, 100]
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()

    replay_metrics = dict(zip(metric_names, map(float, evaluator.stats)))
    maximum_difference = max(
        abs(replay_metrics[name] - expected_metrics[name])
        for name in metric_names
    )
    if maximum_difference > 1e-10:
        raise RuntimeError("Replayed COCOeval differs from the saved metrics.")

    threshold_index = int(np.flatnonzero(
        np.isclose(evaluator.params.iouThrs, 0.5)
    )[0])
    all_area = evaluator.params.areaRng[
        evaluator.params.areaRngLbl.index("all")
    ]

    per_image = {
        image_id: {"TP": 0, "FP": 0, "FN": 0, "matched_gt": set()}
        for image_id in image_ids
    }

    for item in evaluator.evalImgs:
        if item is None or list(item["aRng"]) != list(all_area):
            continue
        if item["maxDet"] != 100:
            continue

        dt_matches = item["dtMatches"][threshold_index]
        dt_ignored = item["dtIgnore"][threshold_index].astype(bool)
        gt_matches = item["gtMatches"][threshold_index]
        gt_ignored = item["gtIgnore"].astype(bool)
        row = per_image[item["image_id"]]

        row["TP"] += int(((dt_matches > 0) & ~dt_ignored).sum())
        row["FP"] += int(((dt_matches == 0) & ~dt_ignored).sum())
        row["FN"] += int(((gt_matches == 0) & ~gt_ignored).sum())
        row["matched_gt"].update(
            int(gt_id)
            for gt_id, matched, ignored in zip(
                item["gtIds"], gt_matches, gt_ignored
            )
            if matched > 0 and not ignored
        )

    return replay_metrics, per_image, maximum_difference

clean_metrics, clean_images, clean_difference = evaluate_saved(
    case_clean_record, baseline["metrics"]
)
faulty_metrics, faulty_images, faulty_difference = evaluate_saved(
    case_fault_record, case_evaluation["metrics"]
)

image_changes = []
for image_id in image_ids:
    clean = clean_images[image_id]
    faulty = faulty_images[image_id]
    image_changes.append({
        "image_id": image_id,
        "clean": {key: clean[key] for key in ("TP", "FP", "FN")},
        "faulty": {key: faulty[key] for key in ("TP", "FP", "FN")},
        "lost_previously_matched_gt_ids": sorted(
            clean["matched_gt"] - faulty["matched_gt"]
        ),
        "newly_matched_gt_ids": sorted(
            faulty["matched_gt"] - clean["matched_gt"]
        ),
    })

totals = {
    label: {
        key: sum(row[key] for row in records.values())
        for key in ("TP", "FP", "FN")
    }
    for label, records in [
        ("clean", clean_images), ("faulty", faulty_images)
    ]
}

report = {
    "status": "EXPLORATORY_CASE_STUDY_NOT_LAYER_RANKING",
    "scenario": case_evaluation["scenario"],
    "selection": "largest_observed_AP_drop_in_incomplete_1579_scenario_checkpoint",
    "image_ids": image_ids,
    "annotation_sha256": split["annotation_sha256"],
    "prediction_sha256": {
        "clean": baseline["prediction_sha256"],
        "faulty": case_evaluation["prediction_sha256"],
    },
    "replayed_metrics": {"clean": clean_metrics, "faulty": faulty_metrics},
    "maximum_replay_metric_difference": {
        "clean": clean_difference, "faulty": faulty_difference,
    },
    "diagnostic_settings": {
        "iou": 0.5,
        "stored_confidence_threshold": 0.001,
        "max_dets": 100,
        "area": "all",
        "crowd_and_ignore_handling": "official_COCOeval",
    },
    "totals": totals,
    "per_image_changes": image_changes,
    "limitations": [
        "Outcome-selected example, not a representative layer estimate.",
        "Diagnostic counts use confidence 0.001, not a deployment threshold.",
        "No inference rerun and no causal attribution to NMS.",
    ],
}

content = (
    json.dumps(report, indent=2, allow_nan=False) + "\n"
).encode("utf-8")
destination.parent.mkdir(parents=True, exist_ok=True)
with destination.open("xb") as handle:
    handle.write(content)
if destination.read_bytes() != content:
    raise RuntimeError("Case-study report verification failed.")

print("All 12 saved COCOeval metrics reproduced: PASS")
print("Clean totals:", totals["clean"])
print("Faulty totals:", totals["faulty"])
print("Previously matched objects lost:", sum(
    len(row["lost_previously_matched_gt_ids"]) for row in image_changes
))
print("Previously unmatched objects gained:", sum(
    len(row["newly_matched_gt_ids"]) for row in image_changes
))
print("Saved:", destination)
print("SHA256:", hashlib.sha256(content).hexdigest())
print("CPU evaluation only. No model inference, commit or push.")

All 12 saved COCOeval metrics reproduced: PASS
Clean totals: {'TP': 1107, 'FP': 16972, 'FN': 204}
Faulty totals: {'TP': 1016, 'FP': 19345, 'FN': 295}
Previously matched objects lost: 118
Previously unmatched objects gained: 27
Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/results/notebook05b/case_studies/S05B-00731_cocoeval_diagnostics_v1.json
SHA256: b7647e686a7aa6aaf82cc48eb65ca64c1ca72458b88441bafece5ba2b52ac35b
CPU evaluation only. No model inference, commit or push.


In [8]:
from pathlib import Path
import hashlib
import json

repo = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
source = (
    repo / "results/notebook05b/case_studies/"
    "S05B-00731_cocoeval_diagnostics_v1.json"
)
content = source.read_bytes()
source_sha = hashlib.sha256(content).hexdigest()

if source_sha != (
    "b7647e686a7aa6aaf82cc48eb65ca64c1ca72458b88441bafece5ba2b52ac35b"
):
    raise RuntimeError("Case-study report SHA256 mismatch.")

report = json.loads(content)
clean = report["totals"]["clean"]
faulty = report["totals"]["faulty"]
lost = sum(
    len(row["lost_previously_matched_gt_ids"])
    for row in report["per_image_changes"]
)
gained = sum(
    len(row["newly_matched_gt_ids"])
    for row in report["per_image_changes"]
)

lines = [
    "# Exploratory fault case: S05B-00731",
    "",
    "Status: verified case study, not a layer-sensitivity ranking.",
    f"Source report SHA256: {source_sha}",
    "",
    "One persistent logical INT8 sign-bit fault in model.0.conv:",
    "q = -69 -> 59. Evaluated on the same 200 development images.",
    "",
    "| Metric | Clean | Faulty | Faulty minus clean |",
    "| --- | ---: | ---: | ---: |",
]

for label, key in [
    ("AP50:95", "AP50_95"),
    ("AP50", "AP50"),
    ("AR100", "AR100"),
]:
    c = report["replayed_metrics"]["clean"][key]
    f = report["replayed_metrics"]["faulty"][key]
    lines.append(f"| {label} | {c:.9f} | {f:.9f} | {f-c:+.9f} |")

lines += [
    "",
    "## Object-level diagnostic",
    "",
    "IoU=0.5; confidence=0.001; COCOeval maxDets=100;",
    "original annotations with official crowd/ignore handling.",
    "",
    "| Count | Clean | Faulty | Difference |",
    "| --- | ---: | ---: | ---: |",
]
for key in ("TP", "FP", "FN"):
    lines.append(
        f"| {key} | {clean[key]} | {faulty[key]} | "
        f"{faulty[key]-clean[key]:+d} |"
    )

lines += [
    "",
    f"Previously matched ground-truth objects lost: {lost}.",
    f"Previously unmatched ground-truth objects gained: {gained}.",
    "",
    "## Evidence and limits",
    "",
    "- Clean and faulty prediction archive/payload checksums were verified.",
    "- All twelve COCOeval metric types were reproduced for both conditions.",
    "- This case was selected for its largest observed AP drop in an incomplete campaign.",
    "- It demonstrates vulnerability of this sampled location, not a definitive layer ranking.",
    "- Counts at confidence 0.001 are not deployment-threshold performance estimates.",
    "- Observed output changes do not establish that NMS caused the degradation.",
    "- No new model inference was performed.",
    "",
]

directory = repo / "docs/research_reports/notebook05b_case_studies_v1"
publications = {
    directory / "S05B-00731_diagnostics_v1.json": content,
    directory / "S05B-00731_case_card_v1.md":
        "\n".join(lines).encode("utf-8"),
}

for path, data in publications.items():
    if path.exists() and path.read_bytes() != data:
        raise RuntimeError(f"Existing publication differs: {path.name}")

directory.mkdir(parents=True, exist_ok=True)
for path, data in publications.items():
    if not path.exists():
        with path.open("xb") as handle:
            handle.write(data)
    if path.read_bytes() != data:
        raise RuntimeError(f"Publication verification failed: {path.name}")
    print("Saved:", path.name)
    print("SHA256:", hashlib.sha256(data).hexdigest())

print("Original report preserved. No inference, staging, commit or push.")

Saved: S05B-00731_diagnostics_v1.json
SHA256: b7647e686a7aa6aaf82cc48eb65ca64c1ca72458b88441bafece5ba2b52ac35b
Saved: S05B-00731_case_card_v1.md
SHA256: 9a00ee88889caf420b45d7252859e47e6ef1c5698aaf13f90491be6ee00826f6
Original report preserved. No inference, staging, commit or push.


In [ ]:
from pathlib import Path
import hashlib
import json
import subprocess

repo = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")

def git(*args):
    return subprocess.run(
        ["git", "-C", str(repo), *args],
        capture_output=True, text=True, check=True,
    ).stdout.strip()

if git("rev-parse", "HEAD") != (
    "1edd3880dcd7d038783119cf56a7365c873aed69"
):
    raise RuntimeError("Unexpected repository HEAD.")
if git("diff", "--cached", "--name-only"):
    raise RuntimeError("Staging area is not empty.")
if git("diff", "--name-only"):
    raise RuntimeError("Tracked modifications require review.")

expected = {
    "docs/research_reports/notebook05b_case_studies_v1/"
    "S05B-00731_diagnostics_v1.json":
        "b7647e686a7aa6aaf82cc48eb65ca64c1ca72458b88441bafece5ba2b52ac35b",
    "docs/research_reports/notebook05b_case_studies_v1/"
    "S05B-00731_case_card_v1.md":
        "9a00ee88889caf420b45d7252859e47e6ef1c5698aaf13f90491be6ee00826f6",
}

for name, expected_sha in expected.items():
    if hashlib.sha256((repo / name).read_bytes()).hexdigest() != expected_sha:
        raise RuntimeError(f"Report identity mismatch: {name}")

candidates = sorted(
    Path("/content/drive/MyDrive/Colab Notebooks").glob("05c*.ipynb")
)
if len(candidates) != 1:
    raise RuntimeError(
        f"Expected one 05c notebook; found: {[str(p) for p in candidates]}"
    )

content = candidates[0].read_bytes()
notebook = json.loads(content)
if notebook.get("nbformat") != 4 or not notebook.get("cells"):
    raise RuntimeError("Invalid or empty notebook.")

error_cells = [
    index
    for index, cell in enumerate(notebook["cells"], start=1)
    if any(
        output.get("output_type") == "error"
        for output in cell.get("outputs", [])
    )
]
if error_cells:
    raise RuntimeError(f"Saved error outputs require review: {error_cells}")

snapshot_name = "notebooks/05c_int8_screening_execution.ipynb"
snapshot = repo / snapshot_name
if snapshot.exists():
    if snapshot.read_bytes() != content:
        raise RuntimeError("Existing notebook snapshot differs.")
else:
    with snapshot.open("xb") as handle:
        handle.write(content)

if snapshot.read_bytes() != content:
    raise RuntimeError("Notebook copy verification failed.")

files = [*expected, snapshot_name]
git("add", "--", *files)
if set(git("diff", "--cached", "--name-only").splitlines()) != set(files):
    raise RuntimeError("Unexpected staged file set.")
git("diff", "--cached", "--check")

print("Notebook cells:", len(notebook["cells"]))
print("Snapshot SHA256:", hashlib.sha256(content).hexdigest())
print("Staged files:", len(files))
print("Whitespace check: PASS")
print("\n" + git("diff", "--cached", "--stat"))
print("\nOriginal notebook preserved. No inference, commit or push.")